# Feature Interaction Maps

Tools for understanding how features interact within the model:
coactivation analysis, mutual suppression/enhancement detection,
feature clustering, interaction strength, and dependency graphs.

## Why This Matters

Feature interaction maps investigates the interpretable directions that models use internally. Understanding features — the natural units of neural computation — is essential for moving beyond neuron-level analysis to a true understanding of what models represent.

**Key references:**
- [Bricken et al. (2023) "Towards Monosemanticity"](https://transformer-circuits.pub/2023/monosemantic-features/index.html) — Sparse autoencoders find interpretable features
- [Cunningham et al. (2023) "Sparse Autoencoders Find Highly Interpretable Features"](https://arxiv.org/abs/2309.08600) — SAE features in larger language models

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.feature_interaction_maps import (
    feature_coactivation,
    mutual_suppression_enhancement,
    feature_clustering,
    interaction_strength,
    feature_dependency_graph,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

tokens_list = [
    jnp.array([1, 2, 3, 4, 5, 6, 7]),
    jnp.array([10, 20, 30, 40, 50, 60, 70]),
    jnp.array([5, 10, 15, 20, 25, 30, 35]),
    jnp.array([2, 4, 6, 8, 10, 12, 14]),
    jnp.array([50, 51, 52, 53, 54, 55, 56]),
    jnp.array([90, 91, 92, 93, 94, 95, 96]),
]

def metric_fn(logits, tokens):
    return jnp.mean(logits[-1])

print("Model and data ready")

## Feature Coactivation

Find dimensions that tend to activate together.

In [ ]:
result = feature_coactivation(model, tokens_list, top_k=5)
print(f"Top coactivating pairs:")
for i, j, corr in result['top_pairs']:
    print(f"  dim {i} <-> dim {j}: correlation={corr:.4f}")
print(f"\nCoactivation matrix shape: {result['coactivation_matrix'].shape}")

## Mutual Suppression & Enhancement

Detect pairs of directions that suppress or enhance each other.

In [ ]:
result = mutual_suppression_enhancement(model, tokens_list, metric_fn, n_directions=4)
print(f"Single direction effects: {[f'{e:.4f}' for e in result['single_effects']]}")
print(f"\nSuppressive pairs: {len(result['suppressive_pairs'])}")
for i, j, v in result['suppressive_pairs'][:3]:
    print(f"  dir {i} <-> dir {j}: interaction={v:.6f}")
print(f"\nEnhancing pairs: {len(result['enhancing_pairs'])}")
for i, j, v in result['enhancing_pairs'][:3]:
    print(f"  dir {i} <-> dir {j}: interaction={v:.6f}")

## Feature Clustering

Cluster residual stream dimensions by coactivation patterns.

In [ ]:
result = feature_clustering(model, tokens_list, n_clusters=4)
print(f"Cluster sizes: {result['cluster_sizes']}")
print(f"Within-cluster correlation: {[f'{c:.3f}' for c in result['within_cluster_correlation']]}")
print(f"Between-cluster correlation: {[f'{c:.3f}' for c in result['between_cluster_correlation']]}")

## Interaction Strength

Measure synergy and redundancy between attention heads.

In [ ]:
result = interaction_strength(model, tokens_list, metric_fn,
                             components=[(0, 0), (0, 1), (1, 0), (1, 1)])
print("Single effects:")
for comp, eff in result['single_effects'].items():
    print(f"  {comp}: {eff:.6f}")
print(f"\nSynergistic pairs: {len(result['synergistic_pairs'])}")
print(f"Redundant pairs: {len(result['redundant_pairs'])}")

## Feature Dependency Graph

Build a graph of activation dimension dependencies.

In [ ]:
result = feature_dependency_graph(model, tokens_list, threshold=0.3)
print(f"Edges: {result['n_edges']}")
print(f"\nHub dimensions (highest degree):")
for dim, degree in result['hub_dimensions'][:5]:
    print(f"  dim {dim}: degree={degree}")